# CBIS-DDSM: Attention-Driven **Weakly-Supervised Lesion Localization**

CBIS-DDSM variant of `mias_attention_localization.ipynb`. Train an EfficientNetB0 + **SE/CBAM**
classifier on **image-level labels only**, then show the attention localizes the lesion - with
two upgrades CBIS enables:

- **Input = full mammogram**, *not* the cropped ROI patch, so localization is non-trivial.
- **Ground truth = the provided binary ROI mask** -> **pixel-precise IoU / Dice**, richer than
  the MIAS `(x,y,radius)` circles.

**Data resolution (important).** The case-description CSVs reference `.dcm` paths, but the images
are JPEGs under `jpeg/<SeriesInstanceUID>/`. We join through **`csv/dicom_info.csv`**
(`SeriesInstanceUID` + `SeriesDescription` -> local jpeg) to pull the *full mammogram* and the
*ROI mask* for each abnormality. Verified locally: full+mask resolve for the sampled rows, masks
are near-binary and share the full-image field-of-view (so independent resize-to-224 stays
aligned).

**Leakage-safe:** CBIS official train/test split is **patient-disjoint**; augmentation on train
only. Label is configurable:

- `TASK='pathology'` -> benign vs malignant (matches your other CBIS work).
- `TASK='abnormality'` -> **mass vs calcification** (well-separated -> the CAM localizes the
  lesion far more reliably; recommended if B/M attention turns out weak).

> **First run:** execute the **diagnostic cell (2b)** to eyeball image<->mask alignment before
> the full training run.


In [ ]:
# ---- GPU setup: PIN to the discrete NVIDIA GPU ----
# TF-DirectML enumerates BOTH the RTX 4050 and the Intel iGPU. The iGPU shares (scarce) system
# RAM and triggers fatal 'CPU->PluggableDevice Memcpy' OOM crashes. Restrict to the NVIDIA GPU.
import os
os.environ.setdefault('DML_VISIBLE_DEVICES', '0')   # RTX=adapter 0; hide Intel iGPU (must precede TF import)
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print('TensorFlow version:', tf.__version__)
print('GPUs detected     :', gpus)
if gpus:
    pick = gpus[0]
    for g in gpus:
        try:
            nm = tf.config.experimental.get_device_details(g).get('device_name', '')
        except Exception:
            nm = ''
        if 'NVIDIA' in nm.upper() or 'RTX' in nm.upper():
            pick = g; break
    try:
        tf.config.set_visible_devices([pick], 'GPU')
        tf.config.experimental.set_memory_growth(pick, True)
        print('Pinned to:', pick)
    except Exception as e:
        print('GPU config error:', e)
else:
    print('\n[WARNING] No GPU visible -> running on CPU.')
print('Visible GPUs:', tf.config.get_visible_devices('GPU'))

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (accuracy_score, roc_auc_score,
                             classification_report, confusion_matrix)
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras import layers
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.applications import EfficientNetB0

In [ ]:
# ============================================================
# CONFIG  -- local (D:) by default; edit for Colab if needed
# ============================================================
DATA_ROOT  = 'D:/cbis-ddsm'                 # Colab: '/content/drive/MyDrive/cbis-ddsm'
JPEG_ROOT  = os.path.join(DATA_ROOT, 'jpeg')
CSV_DIR    = os.path.join(DATA_ROOT, 'csv')
DICOM_INFO = os.path.join(CSV_DIR, 'dicom_info.csv')
SAVE_DIR   = 'G:/My Drive/VS_CC/models'
os.makedirs(SAVE_DIR, exist_ok=True)

TRAIN_CSVS = ['mass_case_description_train_set.csv', 'calc_case_description_train_set.csv']
TEST_CSVS  = ['mass_case_description_test_set.csv',  'calc_case_description_test_set.csv']

TASK        = 'pathology'   # 'pathology' (benign/malignant) | 'abnormality' (mass/calc)
IMG_SIZE    = 224
TAP_LAYER   = 'block6a_activation'
ATTENTION   = 'cbam'        # 'none' | 'se' | 'cbam' | 'se_cbam'
ROT_ANGLES  = [0, 90, 180, 270]
BATCH_SIZE  = 16
EPOCHS      = 30
RANDOM_STATE = 2021
MASK_THRESH = 127

## 1. Resolve full-image + ROI-mask jpeg paths via `dicom_info.csv`

`SeriesInstanceUID` (the folder holding the `.dcm` in the case CSV) -> local jpeg, disambiguated
by `SeriesDescription`. A mask-like fallback covers the rare series whose description is blank or
whose `cropped`/`ROI mask` columns are swapped.

In [ ]:
def _to_local(p):
    p = str(p).replace('\\', '/'); i = p.find('jpeg/')
    return os.path.join(JPEG_ROOT, p[i + 5:]) if i >= 0 else p

def build_uid_maps(dicom_info):
    di = pd.read_csv(dicom_info)
    di['SeriesDescription'] = di['SeriesDescription'].fillna('')
    di['local'] = di['image_path'].apply(_to_local)
    di['uid'] = di['SeriesInstanceUID'].astype(str)
    full, mask, allf = {}, {}, {}
    for _, r in di.iterrows():
        allf.setdefault(r['uid'], []).append(r['local'])
        d = r['SeriesDescription'].lower()
        if 'full' in d:        full.setdefault(r['uid'], r['local'])
        elif 'roi mask' in d:  mask.setdefault(r['uid'], r['local'])
    return full, mask, allf

FULL_MAP, MASK_MAP, ANY_MAP = build_uid_maps(DICOM_INFO)
print('UIDs: full=%d  mask=%d  total=%d' % (len(FULL_MAP), len(MASK_MAP), len(ANY_MAP)))

def series_uid(p):
    parts = str(p).replace('\\', '/').strip().split('/')
    return parts[-2] if len(parts) >= 2 else None

def is_mask_like(g):
    if g is None: return False
    return np.mean((g <= 10) | (g >= 245)) > 0.90 and len(np.unique(g)) <= 64

def resolve_mask(p):
    uid = series_uid(p)
    if uid in MASK_MAP:
        return MASK_MAP[uid]
    for cand in ANY_MAP.get(uid, []):        # fallback: pick the near-binary file
        if is_mask_like(cv2.imread(cand, cv2.IMREAD_GRAYSCALE)):
            return cand
    return None

In [ ]:
def load_tables(names):
    frames = []
    for n in names:
        p = os.path.join(CSV_DIR, n)
        if os.path.exists(p):
            frames.append(pd.read_csv(p))
        else:
            print('  [skip missing]', n)
    df = pd.concat(frames, ignore_index=True)
    df.columns = [c.strip().replace(' ', '_').lower() for c in df.columns]
    return df

def prep(df):
    if TASK == 'pathology':
        df['label'] = (df['pathology'].astype(str).str.upper().str.strip() == 'MALIGNANT').astype(int)
    else:
        src = df['abnormality_type'] if 'abnormality_type' in df.columns else df['abnormality']
        df['label'] = src.astype(str).str.upper().str.contains('MASS').astype(int)
    df['full_path'] = df['image_file_path'].apply(lambda p: FULL_MAP.get(series_uid(p)))
    df['mask_path'] = df['roi_mask_file_path'].apply(resolve_mask)
    df['patient_id'] = df['patient_id'].astype(str)
    df['image_key']  = df['image_file_path'].astype(str)   # groups multi-ROI rows of one mammogram
    return df[['patient_id', 'image_key', 'full_path', 'mask_path', 'label']].dropna(subset=['full_path'])

train_df = prep(load_tables(TRAIN_CSVS))
test_df  = prep(load_tables(TEST_CSVS))

# CBIS mass and calc official splits were built INDEPENDENTLY, so ~31 patients (about 8.5% of
# test rows) appear in both. Enforce patient-disjointness by dropping those patients from TRAIN,
# leaving the official test set intact and honest.
overlap = set(train_df.patient_id) & set(test_df.patient_id)
if overlap:
    before = len(train_df)
    train_df = train_df[~train_df.patient_id.isin(overlap)].reset_index(drop=True)
    print(f'Removed {len(overlap)} overlapping patients from train ({before - len(train_df)} rows).')

print('Train rows:', len(train_df), '| Test rows:', len(test_df))
print('Train label dist:', train_df['label'].value_counts().to_dict())
print('Full resolved: %.0f%% train | Mask resolved: %.0f%% train'
      % (100 * train_df['full_path'].notna().mean(),
         100 * train_df['mask_path'].notna().mean()))
print('Patients disjoint (train vs test):', set(train_df.patient_id).isdisjoint(set(test_df.patient_id)))

## 2. Preprocessing (full image -> CLAHE -> [0,255] 3ch) + mask loading

In [ ]:
_clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

def preprocess_image(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = _clahe.apply(img)
    return np.stack([img] * 3, axis=-1)          # uint8 [0,255]; EfficientNet casts internally (4x less RAM)

def load_mask(path):
    m = cv2.imread(path, cv2.IMREAD_GRAYSCALE) if path else None
    if not is_mask_like(m):
        return None
    m = cv2.resize(m, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
    return (m >= MASK_THRESH).astype(np.uint8)

def rotate(img, angle):
    if angle == 0:
        return img
    code = {90: cv2.ROTATE_90_CLOCKWISE, 180: cv2.ROTATE_180,
            270: cv2.ROTATE_90_COUNTERCLOCKWISE}[angle]
    return cv2.rotate(img, code)

### 2b. DIAGNOSTIC - run FIRST to confirm image<->mask alignment

In [ ]:
probe = test_df[test_df['mask_path'].notna()].drop_duplicates('image_key').head(4)
fig, axes = plt.subplots(1, len(probe), figsize=(3 * len(probe), 3.2), dpi=120)
axes = np.atleast_1d(axes); ok_img = ok_mask = 0
for ax, (_, row) in zip(axes, probe.iterrows()):
    img = preprocess_image(row['full_path']); ok_img += img is not None
    msk = load_mask(row['mask_path']);        ok_mask += msk is not None
    if img is not None:
        ax.imshow(img[..., 0] / 255.0, cmap='gray')
    if msk is not None:
        for c in cv2.findContours(msk, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[-2]:
            c = c.squeeze(1)
            if c.ndim == 2:
                ax.plot(np.append(c[:, 0], c[0, 0]), np.append(c[:, 1], c[0, 1]), color='lime', lw=1.4)
    ax.set_title(('img' if img is not None else 'IMG FAIL') +
                 (' + mask' if msk is not None else ' + NO MASK'), fontsize=8)
    ax.axis('off')
plt.tight_layout(); plt.show()
print(f'images {ok_img}/{len(probe)} | masks {ok_mask}/{len(probe)} '
      '(green contour should sit on the lesion)')

## 3. Build train/test arrays (augment TRAIN only; test clean)

In [ ]:
def build_arrays(df, augment):
    # preallocate a single uint8 array (no list->array doubling); trim unfilled tail
    angles = ROT_ANGLES if augment else [0]
    n = len(df) * len(angles)
    X = np.empty((n, IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
    Y = np.empty(n, dtype=np.int64)
    keys = np.empty(n, dtype=object)
    j = 0
    for row in df.itertuples(index=False):
        base = preprocess_image(row.full_path)
        if base is None:
            continue
        for a in angles:
            X[j] = rotate(base, a); Y[j] = row.label; keys[j] = row.image_key; j += 1
    return X[:j], Y[:j], keys[:j]

x_train, y_train, k_train = build_arrays(train_df, augment=True)
x_test,  y_test,  k_test  = build_arrays(test_df,  augment=False)
print('x_train:', x_train.shape, 'counts', np.bincount(y_train))
print('x_test :', x_test.shape,  'counts', np.bincount(y_test))

# image-key-disjoint validation split (for early stopping) - no leakage across rotations
gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=RANDOM_STATE)
tr_idx, va_idx = next(gss.split(x_train, y_train, groups=k_train))
xt, yt = x_train[tr_idx], y_train[tr_idx]   # advanced indexing already returns independent copies
xv, yv = x_train[va_idx], y_train[va_idx]
# free the full train array (only xt/xv/x_test are used downstream) - low-RAM machines
import gc
del x_train, y_train, k_train, k_test
gc.collect()
print('fit/val:', xt.shape, xv.shape, '| freed x_train')

## 4. Attention blocks (same corrected SE / CBAM as the MIAS notebook)

In [ ]:
def se_block(x, ratio=16):
    f = x.shape[-1]
    s = layers.Reshape((1, 1, f))(layers.GlobalAveragePooling2D()(x))
    s = layers.Dense(max(1, f // ratio), activation='relu', use_bias=False,
                     kernel_initializer='he_normal')(s)
    s = layers.Dense(f, activation='sigmoid', use_bias=False, kernel_initializer='he_normal')(s)
    return layers.Multiply()([x, s])

def cbam_block(x, ratio=16, name='cbam'):
    f = x.shape[-1]
    shared1 = layers.Dense(max(1, f // ratio), activation='relu', kernel_initializer='he_normal')
    shared2 = layers.Dense(f, kernel_initializer='he_normal')
    avg = layers.Reshape((1, 1, f))(layers.GlobalAveragePooling2D()(x))
    mx  = layers.Reshape((1, 1, f))(layers.GlobalMaxPooling2D()(x))
    ch  = layers.Activation('sigmoid')(layers.Add()([shared2(shared1(avg)), shared2(shared1(mx))]))
    x   = layers.Multiply()([x, ch])
    avg_p = layers.Lambda(lambda t: tf.reduce_mean(t, axis=-1, keepdims=True))(x)
    max_p = layers.Lambda(lambda t: tf.reduce_max(t, axis=-1, keepdims=True))(x)
    sp = layers.Concatenate(axis=-1)([avg_p, max_p])
    sp = layers.Conv2D(1, 7, padding='same', activation='sigmoid', name=name + '_spatial')(sp)
    return layers.Multiply()([x, sp])

## 5. Build + train the attention model

In [ ]:
def build_model(attention='cbam', lr=1e-4):
    base = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = False
    x = base.get_layer(TAP_LAYER).output
    if attention == 'se':        x = se_block(x)
    elif attention == 'cbam':    x = cbam_block(x)
    elif attention == 'se_cbam': x = cbam_block(se_block(x))
    x = layers.Activation('linear', name='loc_features')(x)
    g = layers.Dropout(0.5)(layers.GlobalAveragePooling2D(name='gap')(x))
    out = layers.Dense(1, activation='sigmoid')(g)
    m = Model(base.input, out)
    m.compile(optimizer=Adam(lr), loss='binary_crossentropy',
              metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])
    return m

model = build_model(ATTENTION)
cw = compute_class_weight('balanced', classes=np.array([0, 1]), y=yt)
class_weight = {0: cw[0], 1: cw[1]}
es = EarlyStopping(monitor='val_auc', mode='max', patience=6, restore_best_weights=True)
history = model.fit(xt, yt, validation_data=(xv, yv), epochs=EPOCHS, batch_size=BATCH_SIZE,
                    class_weight=class_weight, callbacks=[es], verbose=1)

## 6. Evaluate classification on the official (patient-disjoint) test set

In [ ]:
def auc_ci(y_true, y_prob, n_boot=2000, seed=42):
    rng = np.random.RandomState(seed); y_true = np.asarray(y_true); y_prob = np.asarray(y_prob)
    s = []
    for _ in range(n_boot):
        idx = rng.randint(0, len(y_true), len(y_true))
        if len(np.unique(y_true[idx])) == 2:
            s.append(roc_auc_score(y_true[idx], y_prob[idx]))
    return np.percentile(s, [2.5, 97.5])

prob = model.predict(x_test, batch_size=BATCH_SIZE, verbose=0).ravel()
pred = (prob >= 0.5).astype(int)
cm = confusion_matrix(y_test, pred); tn, fp, fn, tp = cm.ravel()
cnn_acc = accuracy_score(y_test, pred); cnn_auc = roc_auc_score(y_test, prob)
lo, hi = auc_ci(y_test, prob)
pos = 'Malignant' if TASK == 'pathology' else 'Mass'
neg = 'Benign'    if TASK == 'pathology' else 'Calc'
print(f'CNN Accuracy   : {cnn_acc:.4f}')
print(f'CNN ROC-AUC    : {cnn_auc:.4f}  (95% CI {lo:.3f}-{hi:.3f})')
print(f'Sensitivity({pos}): {tp/(tp+fn) if (tp+fn) else 0:.4f}   Specificity({neg}): {tn/(tn+fp) if (tn+fp) else 0:.4f}')
print(classification_report(y_test, pred, target_names=[neg, pos]))

## 7. Weakly-supervised localization vs the ROI mask

Grad-CAM (works for every attention setting) + native CBAM spatial map, scored against the
binary ROI mask: **pointing-game** (peak inside mask), **IoU**, **Dice**, and normalized distance
from the peak to the mask centroid. Masks of multi-ROI mammograms are unioned.

In [ ]:
def gradcam(mdl, arr, layer='loc_features'):
    gm = Model(mdl.inputs, [mdl.get_layer(layer).output, mdl.output])
    with tf.GradientTape() as tape:
        conv, pred = gm(arr); loss = pred[:, 0]
    grads = tape.gradient(loss, conv)
    pooled = tf.reduce_mean(grads, axis=(0, 1, 2))
    heat = tf.nn.relu(tf.reduce_sum(conv[0] * pooled, axis=-1))
    return (heat / (tf.reduce_max(heat) + 1e-8)).numpy()

def cbam_spatial_map(mdl, arr):
    names = [l.name for l in mdl.layers if l.name.endswith('_spatial')]
    if not names:
        return None
    return Model(mdl.inputs, mdl.get_layer(names[0]).output).predict(arr, verbose=0)[0, :, :, 0]

def localization_metrics(mdl, frame, use='gradcam', verbose=False):
    pg, ious, dices, dists = [], [], [], []
    diag = IMG_SIZE * np.sqrt(2)
    for key, rows in frame[frame['mask_path'].notna()].groupby('image_key'):
        base = preprocess_image(rows.iloc[0]['full_path'])
        if base is None:
            continue
        gt = np.zeros((IMG_SIZE, IMG_SIZE), np.uint8)
        for _, r in rows.iterrows():
            m = load_mask(r['mask_path'])
            if m is not None:
                gt |= m
        if gt.sum() == 0:
            continue
        heat = gradcam(mdl, base[None, ...]) if use == 'gradcam' else cbam_spatial_map(mdl, base[None, ...])
        if heat is None:
            return None
        heat = cv2.resize(heat.astype(np.float32), (IMG_SIZE, IMG_SIZE))
        py, px = np.unravel_index(int(np.argmax(heat)), heat.shape)
        pg.append(int(gt[py, px] == 1))
        pm = (heat >= 0.5 * heat.max()).astype(np.uint8)
        inter = np.logical_and(pm, gt).sum(); union = np.logical_or(pm, gt).sum()
        ious.append(inter / union if union else 0.0)
        dices.append(2 * inter / (pm.sum() + gt.sum()) if (pm.sum() + gt.sum()) else 0.0)
        ys, xs = np.where(gt); cy, cx = ys.mean(), xs.mean()
        dists.append(np.hypot(px - cx, py - cy) / diag)
    if not pg:
        return None
    res = dict(n=len(pg), pointing_game=float(np.mean(pg)), iou=float(np.mean(ious)),
               dice=float(np.mean(dices)), centroid_dist=float(np.mean(dists)))
    if verbose:
        print(f"[{use}] n={res['n']} pointing-game={res['pointing_game']:.3f} "
              f"IoU={res['iou']:.3f} Dice={res['dice']:.3f} centroid-dist={res['centroid_dist']:.3f}")
    return res

loc_gradcam = localization_metrics(model, test_df, 'gradcam', verbose=True)
loc_cbam    = localization_metrics(model, test_df, 'cbam',    verbose=True)

### 7b. Qualitative overlays (image + GT mask contour + Grad-CAM)

In [ ]:
ov = test_df[test_df['mask_path'].notna()].drop_duplicates('image_key').head(6)
if len(ov):
    fig, axes = plt.subplots(1, len(ov), figsize=(3 * len(ov), 3.2), dpi=120)
    axes = np.atleast_1d(axes)
    for ax, (_, row) in zip(axes, ov.iterrows()):
        base = preprocess_image(row['full_path']); gt = load_mask(row['mask_path'])
        if base is None:
            ax.axis('off'); continue
        heat = cv2.resize(gradcam(model, base[None, ...]), (IMG_SIZE, IMG_SIZE))
        ax.imshow(base[..., 0] / 255.0, cmap='gray'); ax.imshow(heat, cmap='jet', alpha=0.4)
        if gt is not None:
            for c in cv2.findContours(gt, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[-2]:
                c = c.squeeze(1)
                if c.ndim == 2:
                    ax.plot(np.append(c[:, 0], c[0, 0]), np.append(c[:, 1], c[0, 1]), color='lime', lw=1.4)
        ax.set_title(('malignant' if row['label'] and TASK == 'pathology' else
                      'mass' if row['label'] else 'benign/calc'), fontsize=8)
        ax.axis('off')
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, 'cbis_localization_overlays.png'), dpi=200, bbox_inches='tight')
    plt.show()

## 8. Ablation - attention effect on classification AND localization

In [ ]:
ABL_EPOCHS = 12
rows = []
for att in ['none', 'se', 'cbam', 'se_cbam']:
    m = build_model(att)
    m.fit(xt, yt, validation_data=(xv, yv), epochs=ABL_EPOCHS, batch_size=BATCH_SIZE,
          class_weight=class_weight,
          callbacks=[EarlyStopping(monitor='val_auc', mode='max', patience=4,
                                   restore_best_weights=True)], verbose=0)
    p = m.predict(x_test, verbose=0).ravel()
    lg = localization_metrics(m, test_df, 'gradcam')
    rows.append({'attention': att, 'test_AUC': round(roc_auc_score(y_test, p), 3),
                 'pointing_game': round(lg['pointing_game'], 3) if lg else None,
                 'IoU': round(lg['iou'], 3) if lg else None,
                 'Dice': round(lg['dice'], 3) if lg else None})
    print(rows[-1])
ablation_df = pd.DataFrame(rows)
ablation_df.to_csv(os.path.join(SAVE_DIR, 'cbis_attention_ablation.csv'), index=False)
ablation_df

## 9. Paper-ready summary + save

In [ ]:
lg = loc_gradcam or {'n': 0, 'pointing_game': 0, 'iou': 0, 'dice': 0, 'centroid_dist': 0}
task_txt = 'benign vs. malignant' if TASK == 'pathology' else 'mass vs. calcification'
summary = (
    f"On CBIS-DDSM ({len(train_df)} train / {len(test_df)} test lesion-rows, patient-disjoint "
    f"official split), an EfficientNetB0 backbone with a {ATTENTION.upper()} attention block was "
    f"trained on full mammograms for image-level {task_txt} classification, with train-only "
    f"augmentation to prevent leakage. The model reached ROC-AUC {cnn_auc:.3f} "
    f"(95% CI {lo:.3f}-{hi:.3f}). Without any localization supervision, its Grad-CAM attention "
    f"localized the annotated lesion in {lg['pointing_game']*100:.1f}% of {lg['n']} masked test "
    f"mammograms (pointing-game), with mean attention-vs-ROI IoU {lg['iou']:.3f}, Dice "
    f"{lg['dice']:.3f}, and normalized centroid distance {lg['centroid_dist']:.3f}, validated "
    f"against the CBIS-DDSM ground-truth ROI masks."
)
print(summary)
with open(os.path.join(SAVE_DIR, 'cbis_attention_localization_summary.txt'), 'w') as f:
    f.write(summary)
model.save_weights(os.path.join(SAVE_DIR, f'efficientnet_{ATTENTION}_cbis.weights.h5'))
print('\nSaved weights + summary to', SAVE_DIR)